# Sentinel-2 Change Detection – Duisburg / Oberhausen-West (2018 → 2025)

**Konstantinos Chantzis** · Python · Sentinel-2 L2A · Random Forest

Where did green space disappear between summer 2018 and summer 2025, and what replaced it?

1. Setup, scene selection and loading (Microsoft Planetary Computer, STAC)
2. True-colour images
3. NDVI and NDVI difference
4. Rule-based green loss (drought-robust, water masked, min. 500 m²)
5. Training labels from ESA WorldCover 2021 + SWIR band
6. Random Forest land cover 2018 and 2025
7. What replaced the lost green space?
8. Maps and export (PNG, GeoTIFF, GeoPackage, GeoJSON, CSV)

Runs in Google Colab or any Python 3.10+ environment (see `requirements.txt`).

## 1. Setup, scene selection and loading

In [ ]:
!pip install -q pystac-client odc-stac rioxarray planetary-computer geopandas scikit-learn
import os, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt, xarray as xr
import odc.stac, planetary_computer, rioxarray, geopandas as gpd
from pystac_client import Client
from scipy import ndimage
from pyproj import Transformer
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Untersuchungsgebiet Duisburg / Oberhausen-West (West, Süd, Ost, Nord; WGS84)
bbox = [6.70, 51.38, 6.85, 51.50]
katalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                      modifier=planetary_computer.sign_inplace)

# Wolkenfreieste Szene je Jahr, gleiche Kachel 32ULC
def beste_szene(jahr, monate):
    s = katalog.search(collections=["sentinel-2-l2a"], bbox=bbox,
                       datetime=f"{jahr}-{monate[0]}/{jahr}-{monate[1]}",
                       query={"eo:cloud_cover": {"lt": 5}, "s2:mgrs_tile": {"eq": "32ULC"}})
    it = sorted(s.items(), key=lambda i: i.properties["eo:cloud_cover"])[0]
    print(jahr, "->", it.datetime.date(), "| Wolken:", round(it.properties["eo:cloud_cover"], 2), "%")
    return it

szene_2018 = beste_szene(2018, ["08-01", "08-10"])
szene_2025 = beste_szene(2025, ["07-01", "08-31"])

# Bänder laden (10 m, EPSG:25832), Offset ab Processing Baseline 04.00 korrigieren
def laden(item):
    ds = odc.stac.load([item], bands=["B02", "B03", "B04", "B08"], bbox=bbox,
                       crs="EPSG:25832", resolution=10).isel(time=0).astype("float32")
    offset = 1000 if item.properties.get("s2:processing_baseline", "00") >= "04.00" else 0
    return ((ds - offset) / 10000).clip(0, 1)

bild_2018 = laden(szene_2018)
bild_2025 = laden(szene_2025)
gb = bild_2025.odc.geobox
print("Raster:", bild_2018.sizes["x"], "x", bild_2018.sizes["y"], "Pixel à 10 m")

## 2. True-colour images

In [ ]:
def rgb(ds):
    a = np.dstack([ds.B04, ds.B03, ds.B02])
    return np.clip(a / np.percentile(a, 98), 0, 1)

fig, ax = plt.subplots(1, 2, figsize=(16, 9))
for a, b, t_ in zip(ax, [bild_2018, bild_2025], [str(szene_2018.datetime.date()), str(szene_2025.datetime.date())]):
    a.imshow(rgb(b)); a.set_title("Sentinel-2 – " + t_, fontsize=14); a.axis("off")
plt.tight_layout(); plt.show()

## 3. NDVI and NDVI difference
Summer 2018 was an extreme drought year in Germany: grassland was dry, so a simple NDVI difference shows mostly *greening* and is not a valid change indicator on its own.

In [ ]:
def ndvi(ds):
    return ((ds.B08 - ds.B04) / (ds.B08 + ds.B04 + 1e-6)).clip(-1, 1)

ndvi_2018, ndvi_2025 = ndvi(bild_2018), ndvi(bild_2025)
diff = ndvi_2025 - ndvi_2018

fig, ax = plt.subplots(1, 3, figsize=(20, 8))
for a, d, t_ in zip(ax[:2], [ndvi_2018, ndvi_2025], ["NDVI 2018", "NDVI 2025"]):
    im = a.imshow(d, cmap="RdYlGn", vmin=-0.2, vmax=0.9); a.set_title(t_, fontsize=14); a.axis("off")
plt.colorbar(im, ax=ax[:2], shrink=0.6, label="NDVI")
im2 = ax[2].imshow(diff, cmap="RdBu", vmin=-0.5, vmax=0.5)
ax[2].set_title("Differenz 2025 − 2018", fontsize=14); ax[2].axis("off")
plt.colorbar(im2, ax=ax[2], shrink=0.6, label="NDVI-Änderung"); plt.show()
print("Mittlerer NDVI 2018:", round(float(ndvi_2018.mean()), 3), "| 2025:", round(float(ndvi_2025.mean()), 3))

## 4. Rule-based green loss
A pixel counts as green loss if it was **dense vegetation in 2018 (NDVI ≥ 0.5, i.e. green even during the drought)** and is **sealed or bare in 2025 (NDVI < 0.25)**. Water (NDWI > 0 in either year) is excluded and patches smaller than 5 pixels (500 m²) are removed.

In [ ]:
def ndwi(ds):
    return (ds.B03 - ds.B08) / (ds.B03 + ds.B08 + 1e-6)
wasser = (ndwi(bild_2018) > 0) | (ndwi(bild_2025) > 0)

verlust = ((ndvi_2018 >= 0.5) & (ndvi_2025 < 0.25) & ~wasser).values
label, n = ndimage.label(verlust)
groesse = ndimage.sum(verlust, label, range(1, n + 1))
behalten = np.isin(label, np.where(groesse >= 5)[0] + 1)
label, n = ndimage.label(behalten)
groesse = ndimage.sum(behalten, label, range(1, n + 1))

print(f"Grünverlust 2018 → 2025: {behalten.sum() / 100:.1f} ha in {n} Flächen")
print(f"Wasserfläche ausgeschlossen: {wasser.values.sum() / 100:.0f} ha")

t = Transformer.from_crs(25832, 4326, always_xy=True)
xs, ys = bild_2025.x.values, bild_2025.y.values
top = np.argsort(groesse)[::-1][:10]
for rang, (i, (r, c)) in enumerate(zip(top, ndimage.center_of_mass(behalten, label, top + 1)), 1):
    lon, lat = t.transform(xs[int(c)], ys[int(r)])
    print(f"{rang:2d}. {groesse[i] / 100:5.2f} ha   {lat:.5f}, {lon:.5f}")

## 5. Training labels (ESA WorldCover 2021) and SWIR band

In [ ]:
wc_item = next(katalog.search(collections=["esa-worldcover"], bbox=bbox, datetime="2021").items())
wc = odc.stac.load([wc_item], bands=["map"], geobox=gb).isel(time=0)["map"].values

# 1 Bäume | 2 Grünland/Acker | 3 Bebaut | 4 Offener Boden | 5 Wasser
umsetzung = {10: 1, 20: 1, 95: 1, 30: 2, 40: 2, 90: 2, 100: 2, 50: 3, 60: 4, 80: 5}
labels = np.zeros_like(wc, dtype="uint8")
for k, v in umsetzung.items():
    labels[wc == k] = v
KLASSEN = ["Bäume", "Grünland/Acker", "Bebaut", "Offener Boden", "Wasser"]
FARBEN = ["#1a7a2e", "#a6d96a", "#d7301f", "#e6c88a", "#2b6cb0"]
cmap = ListedColormap(FARBEN)

def swir(item):
    b = odc.stac.load([item], bands=["B11"], geobox=gb).isel(time=0)["B11"].astype("float32")
    offset = 1000 if item.properties.get("s2:processing_baseline", "00") >= "04.00" else 0
    return ((b - offset) / 10000).clip(0, 1)
bild_2018["B11"] = swir(szene_2018)
bild_2025["B11"] = swir(szene_2025)

for i, k in enumerate(KLASSEN, 1):
    print(f"{k:16s} {np.sum(labels == i) / 100:8.0f} ha")

## 6. Random Forest land cover 2018 and 2025
One model per year (so the 2018 model learns that *dry* grass is still grassland). 8 features: B02, B03, B04, B08, B11, NDVI, NDWI, NDBI. Up to 3,000 samples per class, 70/30 train/test split.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

def merkmale(ds):
    nd = lambda a, b: (a - b) / (a + b + 1e-6)
    f = [ds.B02, ds.B03, ds.B04, ds.B08, ds.B11, nd(ds.B08, ds.B04), nd(ds.B03, ds.B08), nd(ds.B11, ds.B08)]
    return np.stack([x.values for x in f], axis=-1)

rng = np.random.default_rng(42)
idx = np.concatenate([rng.choice(np.flatnonzero(labels.ravel() == k),
                                 size=min(3000, int(np.sum(labels == k))), replace=False) for k in range(1, 6)])
y = labels.ravel()[idx]
karten = {}
for jahr, ds in [(2018, bild_2018), (2025, bild_2025)]:
    F = merkmale(ds)
    X_tr, X_te, y_tr, y_te = train_test_split(F.reshape(-1, 8)[idx], y, test_size=0.3, stratify=y, random_state=42)
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42).fit(X_tr, y_tr)
    print(f"\n=== {jahr} – Genauigkeit (Testdaten): {accuracy_score(y_te, rf.predict(X_te)):.1%}")
    print(classification_report(y_te, rf.predict(X_te), target_names=KLASSEN, digits=2))
    karten[jahr] = rf.predict(F.reshape(-1, 8)).reshape(F.shape[:2])

tab = pd.DataFrame({j: [np.sum(karten[j] == k) / 100 for k in range(1, 6)] for j in [2018, 2025]}, index=KLASSEN).round(0)
tab["Änderung (ha)"] = tab[2025] - tab[2018]
print(tab)   # Achtung: Differenz zweier unabhängiger Klassifikationen ist NICHT als Änderung interpretierbar

## 7. What replaced the lost green space?

In [ ]:
k25 = karten[2025]
for i, k in enumerate(KLASSEN, 1):
    ha = np.sum(behalten & (k25 == i)) / 100
    if ha > 0:
        print(f"   {k:16s} {ha:6.1f} ha  ({ha / (behalten.sum() / 100):5.1%})")

zeilen = []
for nr in range(1, n + 1):
    maske = label == nr
    dom = np.bincount(k25[maske], minlength=6)[1:].argmax() + 1
    r, c = ndimage.center_of_mass(maske)
    lon, lat = t.transform(xs[int(c)], ys[int(r)])
    zeilen.append({"Fläche_Nr": nr, "Fläche_ha": maske.sum() / 100, "Landbedeckung_2025": KLASSEN[dom - 1],
                   "NDVI_2018": float(ndvi_2018.values[maske].mean()), "NDVI_2025": float(ndvi_2025.values[maske].mean()),
                   "Breite": lat, "Länge": lon})
flaechen = pd.DataFrame(zeilen).sort_values("Fläche_ha", ascending=False).reset_index(drop=True)
print(flaechen["Landbedeckung_2025"].value_counts())
print(flaechen.head(10).round(3).to_string(index=False))
farbe_verlust = np.where(behalten, k25, 0)

## 8. Maps and export

In [ ]:
from rasterio.features import shapes
from shapely.geometry import shape

OUT = "ergebnisse"; os.makedirs(OUT, exist_ok=True)
d18, d25 = str(szene_2018.datetime.date()), str(szene_2025.datetime.date())
quelle = "Daten: Copernicus Sentinel-2 L2A (ESA), Microsoft Planetary Computer · ESA WorldCover 2021 · EPSG:25832"

fig, ax = plt.subplots(1, 2, figsize=(18, 11))
for a, b, t_ in zip(ax, [bild_2018, bild_2025], [d18, d25]):
    a.imshow(rgb(b)); a.set_title(f"Sentinel-2 Echtfarben – {t_}", fontsize=15); a.axis("off")
fig.suptitle("Duisburg / Oberhausen-West – vorher und nachher", fontsize=18, fontweight="bold")
fig.text(0.01, 0.01, quelle, fontsize=9); plt.tight_layout(rect=[0, 0.02, 1, 0.97])
plt.savefig(f"{OUT}/01_Vorher_Nachher.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(12, 14))
ax.imshow(rgb(bild_2025) * 0.5 + 0.5)
ax.imshow(np.ma.masked_equal(farbe_verlust, 0), cmap=cmap, vmin=1, vmax=5, interpolation="nearest")
ax.set_title(f"Grünverlust {d18[:4]} → {d25[:4]}: {behalten.sum()/100:.1f} ha in {n} Flächen", fontsize=16, fontweight="bold")
ax.legend(handles=[Patch(color=FARBEN[2], label="→ Bebaut"), Patch(color=FARBEN[3], label="→ Offener Boden / Baustelle"),
                   Patch(color=FARBEN[1], label="→ Grünland/Acker")], loc="lower left", fontsize=11)
ax.axis("off"); fig.text(0.01, 0.01, quelle, fontsize=9)
plt.savefig(f"{OUT}/02_Gruenverlust.png", dpi=200, bbox_inches="tight"); plt.show()

beispiele = [(145, "A40-Ausbau bei Amazon DNW4"), (159, "Baustelle AS DU-Homberg (A40)"), (46, "Geräumte Fläche, Oberhausen")]
rgb18, rgb25 = rgb(bild_2018), rgb(bild_2025)
fig, ax = plt.subplots(3, 2, figsize=(12, 18))
for z, (nr, titel) in enumerate(beispiele):
    r, c = [int(v) for v in ndimage.center_of_mass(label == nr)]
    sl = (slice(max(r - 40, 0), r + 40), slice(max(c - 40, 0), c + 40))
    for s, (bild, jahr) in enumerate([(rgb18, d18), (rgb25, d25)]):
        ax[z, s].imshow(bild[sl]); ax[z, s].contour(label[sl] == nr, levels=[0.5], colors="yellow", linewidths=2)
        ax[z, s].set_title(f"{titel}\n{jahr}", fontsize=12); ax[z, s].axis("off")
fig.suptitle("Geprüfte Beispiele (Ausschnitt 800 × 800 m, gelb = erkannte Fläche)", fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.savefig(f"{OUT}/03_Zoom_Beispiele.png", dpi=200); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(18, 11))
for a, j in zip(ax, [2018, 2025]):
    a.imshow(karten[j], cmap=cmap, vmin=1, vmax=5, interpolation="nearest"); a.set_title(f"Landbedeckung {j} (Random Forest)", fontsize=15); a.axis("off")
ax[1].legend(handles=[Patch(color=f, label=k) for k, f in zip(KLASSEN, FARBEN)], loc="lower left", fontsize=11)
fig.text(0.01, 0.01, quelle, fontsize=9); plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(f"{OUT}/04_Landbedeckung.png", dpi=200); plt.show()

def als_raster(arr, name, dtype="uint8"):
    da = xr.DataArray(arr.astype(dtype), coords={"y": bild_2025.y, "x": bild_2025.x}, dims=("y", "x"))
    da.rio.write_crs("EPSG:25832", inplace=True).rio.to_raster(f"{OUT}/{name}", compress="deflate")
als_raster(behalten, "gruenverlust_2018_2025.tif")
als_raster(karten[2018], "landbedeckung_2018.tif")
als_raster(karten[2025], "landbedeckung_2025.tif")

geoms = [(int(v), shape(g)) for g, v in shapes(label.astype("int32"), mask=label > 0, transform=gb.affine)]
gdf = gpd.GeoDataFrame({"Fläche_Nr": [v for v, _ in geoms]}, geometry=[g for _, g in geoms], crs="EPSG:25832")
gdf = gdf.dissolve(by="Fläche_Nr").reset_index().merge(flaechen, on="Fläche_Nr").sort_values("Fläche_ha", ascending=False)
gdf.round(4).to_file(f"{OUT}/gruenverlust_flaechen.gpkg", driver="GPKG")
gdf.to_crs(4326).round(4).to_file(f"{OUT}/gruenverlust_flaechen.geojson", driver="GeoJSON")
flaechen.round(4).to_csv(f"{OUT}/gruenverlust_flaechen.csv", index=False)

shutil.make_archive("Sentinel2_Ergebnisse", "zip", OUT)
try:
    from google.colab import files
    files.download("Sentinel2_Ergebnisse.zip")
except ImportError:
    print("Ergebnisse gespeichert in:", os.path.abspath(OUT))